# **Random Forest**

This notebook evaluates Random Forest as the second supervised machine learning model for HDFS anomaly detection.

Random Forest combines multiple decision trees and uses their collective predictions to classify each HDFS block as normal or anomalous. Unlike Logistic Regression, which learns a linear relationship between the engineered features and the target label, Random Forest can capture more complex and nonlinear relationships within the data.

Each of the four engineered feature sets is evaluated independently using the same training and testing splits created during model preparation. Model performance is assessed using Accuracy, Precision, Recall, and F1 Score to allow direct comparison with the statistical rule based baseline and Logistic Regression.

## Environment Setup

The required libraries are imported, and the project directory structure is initialized. The prepared training and testing datasets generated during model preparation will be loaded from the `data/processed/splits/` directory.

In [1]:
# Import required libraries

from pathlib import Path

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [2]:
# Determine the project root directory

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

processed_dir = project_root / "data" / "processed"
splits_dir = processed_dir / "splits"

results_dir = project_root / "results"
random_forest_results_dir = results_dir / "random_forest"

random_forest_results_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Splits folder:", splits_dir)
print("Results folder:", random_forest_results_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Splits folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\splits
Results folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\random_forest


## Load Prepared Training and Testing Data

The training and testing datasets created during the model preparation stage are loaded for each engineered feature set. Using the same train-test split ensures that Random Forest is evaluated under identical conditions as the statistical rule based baseline and Logistic Regression, allowing for a fair comparison of model performance.

In [3]:
# Load the prepared training and testing datasets for each feature set

random_forest_data = {}

for i in range(1, 5):

    feature_dir = splits_dir / f"feature_set_{i}"

    random_forest_data[f"feature_set_{i}"] = {
        "X_train": pd.read_csv(feature_dir / "X_train.csv"),
        "X_test": pd.read_csv(feature_dir / "X_test.csv"),
        "y_train": pd.read_csv(feature_dir / "y_train.csv").squeeze(),
        "y_test": pd.read_csv(feature_dir / "y_test.csv").squeeze(),
    }

In [4]:
# Verify the loaded dataset dimensions

for name, data in random_forest_data.items():

    print(f"\n{name}")

    print(f"X_train: {data['X_train'].shape}")
    print(f"X_test : {data['X_test'].shape}")
    print(f"y_train: {data['y_train'].shape}")
    print(f"y_test : {data['y_test'].shape}")


feature_set_1
X_train: (460048, 29)
X_test : (115013, 29)
y_train: (460048,)
y_test : (115013,)

feature_set_2
X_train: (460048, 31)
X_test : (115013, 31)
y_train: (460048,)
y_test : (115013,)

feature_set_3
X_train: (460048, 311)
X_test : (115013, 311)
y_train: (460048,)
y_test : (115013,)

feature_set_4
X_train: (460048, 1082)
X_test : (115013, 1082)
y_train: (460048,)
y_test : (115013,)


## Train Random Forest Models

A separate Random Forest model is trained for each engineered feature set using the prepared training data.

Random Forest is an ensemble learning algorithm that combines the predictions of multiple decision trees. During training, each tree is built from a different random sample of the training data and a random subset of the engineered features. The final prediction is determined by majority vote across all trees.

Because the HDFS dataset is highly imbalanced, balanced class weights are used during training to place greater emphasis on correctly identifying the minority anomalous class.

In [5]:
# Train a Random Forest model for each engineered feature set

# Create a dictionary to store the trained Random Forest models
random_forest_models = {}

# Train one model for each feature set
for name, data in random_forest_data.items():

    # Retrieve the training features and labels
    X_train = data["X_train"]
    y_train = data["y_train"]

    # Create the Random Forest model
    # n_estimators=100 builds 100 decision trees.
    # class_weight="balanced" accounts for the class imbalance.
    # random_state=42 ensures reproducible results.
    model = RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42
    )

    # Train the model
    model.fit(X_train, y_train)

    # Store the trained model
    random_forest_models[name] = model

In [6]:
# Confirm that all Random Forest models were successfully trained

for name in random_forest_models:

    print(f"{name}: Model trained successfully")

feature_set_1: Model trained successfully
feature_set_2: Model trained successfully
feature_set_3: Model trained successfully
feature_set_4: Model trained successfully


## Evaluate Random Forest Performance

Each trained Random Forest model is evaluated using the corresponding testing dataset.

Predictions are generated for every HDFS block in the testing set and compared with the ground-truth labels. Performance is measured using Accuracy, Precision, Recall, and F1 Score, allowing direct comparison with the statistical rule based baseline and Logistic Regression.

In [7]:
# Evaluate the performance of each Random Forest model

# Create a list to store the evaluation results
random_forest_results = []

# Evaluate one model for each feature set
for name, model in random_forest_models.items():

    # Retrieve the testing features and labels
    X_test = random_forest_data[name]["X_test"]
    y_test = random_forest_data[name]["y_test"]

    # Predict the class label for each testing block
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    random_forest_results.append({
        "Feature_Set": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    })

# Convert the results into a DataFrame
random_forest_results = pd.DataFrame(random_forest_results)

# Display the evaluation results
display(random_forest_results)

,Feature_Set,Accuracy,Precision,Recall,F1
0,feature_set_1,0.999930,0.998221,0.999406,0.998813
1,feature_set_2,0.999930,0.998221,0.999406,0.998813
2,feature_set_3,0.999835,0.995561,0.998812,0.997184
3,feature_set_4,0.999826,0.995560,0.998515,0.997035


In [8]:
# Export the Random Forest results

random_forest_results_path = (
    random_forest_results_dir / "random_forest_results.csv"
)

random_forest_results.to_csv(
    random_forest_results_path,
    index=False
)

print("Random Forest results saved to:")
print(random_forest_results_path)

Random Forest results saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\random_forest\random_forest_results.csv


# Random Forest Summary

Random Forest was successfully trained and evaluated across all four engineered feature sets. Compared to the statistical rule based baseline, Random Forest achieved a substantial improvement in anomaly detection performance. The highest statistical baseline F1 score was **0.4125**, whereas Random Forest achieved a best F1 score of **0.9988** using **Feature Set 1** and **Feature Set 2**, representing an improvement of approximately **142%**.

Unlike Logistic Regression, additional engineered features did not improve Random Forest performance. The two simplest feature sets produced the highest F1 score (**0.9988**), while Feature Sets 3 and 4 showed a slight decrease in performance (**0.9972** and **0.9970**), respectively. This suggests that the original event occurrence features contained sufficient information for Random Forest to accurately distinguish between normal and anomalous HDFS blocks, and that the additional engineered sequential features provided little additional predictive value.

Overall, Random Forest produced the strongest performance observed thus far, achieving nearly perfect classification across all evaluation metrics while demonstrating that increasingly complex feature engineering was not necessary for this model.